In [1]:
#Считайте данные из файлов, сохраните их в переменные соответствующие переменные

import pandas as pd

df_p = pd.read_csv("Products.csv", sep=';') # для Products
df_c = pd.read_csv("Customers.csv", sep=';') # для Customers
df_o = pd.read_csv("Orders.csv", sep=';')  # для Orders
df_od = pd.read_csv("Order_details.csv", sep=';') # для Order_details
df_e = pd.read_excel("Employees.xlsx") # для Employees

In [ ]:
#по данным из файлов Orders и Employees
Как зовут сотрудника, который оформил больше всего заказов за все время? #Margaret Peacock

In [5]:

df_comb = df_o.merge(df_e)
counts = df_comb.groupby(['FirstName','LastName'])['OrderID'].count()
top_name = counts.idxmax()
print(top_name[0] + ' ' + top_name[1])

Margaret Peacock


In [ ]:
#по данным из файлов Orders, Customers и Order_details
Как зовут клиента, который принес самую высокую чистую выручку за все время? #Horst Kloss

In [ ]:
df_comb = df_o.merge(df_c).merge(df_od)
df_comb['NetRevenue'] = df_comb['UnitPrice'] * df_comb['Quantity'] * (1 - df_comb['Discount'])

display(df_comb.groupby('ContactName')['NetRevenue'].sum().idxmax()) 

'Horst Kloss'

In [ ]:
#по данным из файлов Products и Order_details
Какая категория товаров принесла самую высокую чистую выручку? #Beverages

нюанс:

В таблицах Products и Order_details есть общий ключ ProductID.
Но также в этих таблицах есть еще один столбец с одинаковым именем UnitPrice.
Причем UnitPrice в df_od - это стоимость продукта в момент оформления заказа,
а UnitPrice в df_p - это стоимость продукта на «сегодняшний день».
Стоимости продуктов в df_od и в df_p могут различаться.

In [49]:
df_comb = df_od.merge(df_p, on='ProductID', suffixes=('_order', '_product'))
df_comb['NetRevenue'] = df_comb['UnitPrice_product'] * df_comb['Quantity'] * (1 - df_comb['Discount'])
display(df_comb.groupby('CategoryName')['NetRevenue'].sum().idxmax())

'Beverages'

In [ ]:
#по данным из файлов Products и Order_details
Какую чистую выручку принесла категория Confections? #167357.2250

нюанс:

В таблицах Products и Order_details есть общий ключ ProductID.
Но также в этих таблицах есть еще один столбец с одинаковым именем UnitPrice.
Причем UnitPrice в df_od - это стоимость продукта в момент оформления заказа,
а UnitPrice в df_p - это стоимость продукта на «сегодняшний день».
Стоимости продуктов в df_od и в df_p могут различаться.

In [78]:
df_comb = df_o.merge(df_od, on='OrderID').merge(df_p, on='ProductID', suffixes=('_order', '_product'))
df_comb['NetRevenue'] = df_comb['UnitPrice_order'] * df_comb['Quantity'] * (1 - df_comb['Discount'])
print(df_comb[df_comb['CategoryName'] == 'Confections']['NetRevenue'].sum().round(3))

167357.225


In [ ]:
#по данным из файлов Customers и Employees
C помощью метода .concat() объедините должности сотрудников и должности клиентов. Сколько человек состоят в должности Sales Representative? #23

In [ ]:
df_con = pd.concat([df_c['ContactTitle'], df_e['Title']])
print((df_con == 'Sales Representative').sum())

23


In [ ]:
#по данным из файла Orders
В каком месяце было совершено рекордное количество заказов? #04 - 1998

In [100]:
display(df_o['OrderDate'].str[:7].value_counts().idxmax())

'1998-04'

In [ ]:
#по данным из файла Orders
Сколько заказов было сделано на 43 неделе 1996 года? #5

In [135]:
df_o['OrderDate'] = pd.to_datetime(df_o['OrderDate'])

display(df_o.loc[df_o['OrderDate'].between(pd.to_datetime('1996') + pd.to_timedelta(43, unit='W'), pd.to_datetime('1996') + pd.to_timedelta(43, unit='W') + pd.to_timedelta(7, unit='D'))].shape[0])

7

In [ ]:
#по данным из файла Orders и Order_details
Посчитайте рекордную чистую выручку, которую сделал один сотрудник в месяц? #30990,28

In [144]:
df_comb = df_o.merge(df_o).merge(df_od)
df_comb['NetRevenue'] = df_comb['UnitPrice'] * df_comb['Quantity'] * (1 - df_comb['Discount'])
print(df_comb.groupby([df_comb['EmployeeID'], df_comb['OrderDate'].dt.to_period('M')])['NetRevenue'].sum().max())

30990.28


In [ ]:
#по данным из файла Orders
С помощью метода .pivot_table() постройте сводную таблицу, в которой по вертикали будут годы, а по горизонтали кварталы.
Посчитайте количество заказов. Сколько заказов было оформлено в 1 квартале 1998 года? #182

In [9]:
od = pd.to_datetime(df_o['OrderDate'])
pd.pivot_table(df_o,
               values='OrderID',
               index=od.dt.year,
               columns=od.dt.quarter,
               aggfunc='count',
               fill_value=0)

OrderDate,1,2,3,4
OrderDate,,,,
1996,0,0,70,82
1997,92,93,103,120
1998,182,88,0,0


In [ ]:
#по данным из файлов Orders и Employees

Создайте датафрейм df_sex в котором будет два столбца:

TitleOfCourtesy	  Sex
Ms.	              female
Dr.	              male
Mrs.	            female
Mr.	              male

Присоедините к датафрейму с сотрудниками df_e датафрейм df_sex.
Далее в разрезе пола посчитайте количество оформленных заказов.
На сколько представители одного пола оформили больше заказов, чем представители другого пола? #на 276

In [15]:
df_sex = pd.DataFrame({
    'TitleOfCourtesy': ['Ms.', 'Dr.', 'Mrs.', 'Mr.'],
    'Sex': ['female', 'male', 'female', 'male']
})  
df_comb = df_e.merge(df_sex, on='TitleOfCourtesy')
df_comb = df_comb.merge(df_o, on='EmployeeID')
order_counts = df_comb.groupby('Sex')['OrderID'].count()
print(abs(order_counts['female'] - order_counts['male']))

276


In [ ]:
#по данным из файла Order_details
Посчитайте для каждого заказа чистую выручку.
Затем с помощью метода .cut() разбейте столбец с чистой выручкой на 5 бинов.
Добавьте столбец с бинами в датафрейм. Далее в разрезе бинов посчитайте количество заказов. #750 - 60 - 9 - 2

In [10]:
df_od['NetRevenue'] = df_od['UnitPrice'] * df_od['Quantity'] * (1 - df_od['Discount'])

order_rev = df_od.groupby('OrderID', as_index=False)['NetRevenue'].sum()

order_rev['RevenueBin'] = pd.cut(order_rev['NetRevenue'], bins=5)

print(order_rev['RevenueBin'].value_counts().sort_index())

RevenueBin
(-3.875, 3287.5]      750
(3287.5, 6562.5]       60
(6562.5, 9837.5]        9
(9837.5, 13112.5]       9
(13112.5, 16387.5]      2
Name: count, dtype: int64


In [ ]:
#по данным из файла Customers
Используя слайсинг выведете строки с 7 по 13

In [ ]:
display(df_c[7:14])

,CustomerID,ContactName,ContactTitle,Phone,Address
7,BOLID,Martin Sommer,Owner,(91) 555 22 82,"C/ Araquil, 67"
8,BONAP,Laurence Lebihan,Owner,91.24.45.40,"12, rue des Bouchers"
9,BOTTM,Elizabeth Lincoln,Accounting Manager,(604) 555-4729,23 Tsawassen Blvd.
10,BSBEV,Victoria Ashworth,Sales Representative,(171) 555-1212,Fauntleroy Circus
11,CACTU,Patricio Simpson,Sales Agent,(1) 135-5555,Cerrito 333
12,CENTC,Francisco Chang,Marketing Manager,(5) 555-3392,Sierras de Granada 9993
13,CHOPS,Yang Wang,Owner,0452-076545,Hauptstr. 29


In [ ]:
#
Вызовите метод .apply() для столбца ContactTitle и примените функцию grouping_by_profession() для каждого значения стобца.
Функция grouping_by_profession() должна возвращать 'Sales' если в нее передана строка, содержащая 'Sales',
возвращать 'Marketing' для строк с этим словом и 'Other' для всех остальных.
Посчитайте кол-во кастомеров для каждой группы профессий. #Sales 43, Marketing 19, Other 29

In [57]:
def grouping_by_profession(title):
    if 'Sales' in title:
        return 'Sales'
    elif 'Marketing' in title:
        return 'Marketing'
    else:
        return 'Other'

df_grouped = df_c['ContactTitle'].apply(grouping_by_profession)
print("Sales =", df_grouped[df_grouped == 'Sales'].count())
print("Marketing =", df_grouped[df_grouped == 'Marketing'].count())
print("Other =", df_grouped[df_grouped == 'Other'].count())


Sales = 43
Marketing = 19
Other = 29
